# Trial Count Sweep Analysis

This notebook measures how classification accuracy (ts+FGDA) changes as a function of the number of training trials.

It answers: **how many trials are needed to reach peak performance, and at what point do more trials yield diminishing returns?**

### Approach
Within each CV fold, we draw progressively larger stratified subsamples from the training set, train the classifier on each, and evaluate on the same held-out test trials. Repeating this across all folds and multiple random draws gives a robust **learning curve**: mean accuracy ± SEM vs. number of training trials.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

import json
import pathlib
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedShuffleSplit

from src.preprocessing import *
from src.training import *
from src.evaluation import *
from src.group_analysis import *

## 1. Load Preprocessed Data

Fill in the subject name and the path to your preprocessed epochs below.  
The cells mirror the standard preprocessing flow from `Main.ipynb`.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
SUBJECT_NAME = 'YourSubjectName'   # <-- fill in
PIPELINE_NAME = 'ts+FGDA'

# Sweep parameters
N_TRIALS_LIST = [5, 10, 20, 30, 40, 60, 80, 100, 120]  # trial counts to test
N_REPEATS = 10     # random draws per (fold, n_trials); more = smoother curve
TMIN = 0.0         # evaluation window start (seconds, relative to event onset)
TMAX = 5.0         # evaluation window end

METRICS_DIR = pathlib.Path('..') / 'Metrics'
METRICS_DIR.mkdir(exist_ok=True)

In [ ]:
# ── params_dict ──────────────────────────────────────────────────────────────
# Copy the relevant params_dict block from your Main.ipynb for this subject.
# Minimum required keys are shown below; extend as needed.
params_dict = {}
params_dict['pipeline_name'] = PIPELINE_NAME
params_dict['desired_events'] = ['ClosePalm', 'RightHand', 'LeftHand']  # adjust
params_dict['epoch_tmin'] = -5    # adjust to your epoch crop start
params_dict['CenterByClass'] = True   # per-class mean removal in EEG_Preprocessing;
                                      # label-dependent, so the live loop cannot apply it
params_dict['augmentation_params'] = {'win_len': 0, 'win_step': 0.25}
params_dict['windowed_prediction_params'] = {'win_len': 2, 'win_step': 0.25}
# params_dict['events_trigger_dict'] will be filled after preprocessing below
# params_dict['n_components'] = 4  # only needed for CSP-based pipelines

In [ ]:
# ── Load epochs ───────────────────────────────────────────────────────────────
# Replace this cell with your actual data loading (e.g. from Main.ipynb).
# After this cell, `epochs` and `epochs_cropped` must exist as mne.Epochs objects,
# and params_dict['events_trigger_dict'] must be set.
#
# Example (adapt as needed):
#   current_path = pathlib.Path('..').resolve()
#   recording_path = current_path / 'Data' / SUBJECT_NAME
#   xdf_files = list(recording_path.glob('*.xdf'))
#   raw = read_raw_xdf(xdf_files[0])
#   filtered_raw, epoch, filter_bank_epochs, mean_across_epochs, events_trigger_dict = \
#       EEG_Preprocessing(current_path, raw, params_dict, pick_channels=True)
#   epoch = remap_epoch_events_to_standard(epoch, standard_event_id, params_dict['desired_events'])
#   params_dict['events_trigger_dict'] = events_trigger_dict
#
#   epochs_cropped = epoch.copy().crop(tmin=params_dict['classifier_window_s'],
#                                      tmax=params_dict['classifier_window_e'])
#   epochs = epoch  # uncropped

raise NotImplementedError("Fill in your data loading code above then remove this line.")

## 2. Set Up CV Split

We use the same 10-fold `StratifiedShuffleSplit` as the main pipeline.

In [ ]:
labels = epochs_cropped.events[:, -1]
cv = StratifiedShuffleSplit(n_splits=10, test_size=0.2, random_state=42)
cv_split = list(cv.split(np.zeros(len(labels)), labels))

# Report how many training trials are available in the largest fold
max_train = max(len(tr) for tr, _ in cv_split)
print(f"Total trials: {len(labels)}")
print(f"Max training trials per fold: {max_train}")
print(f"Sweep will test n_trials = {N_TRIALS_LIST}")

# Trim list to avoid going above max_train (keeps informative points only)
N_TRIALS_LIST = [n for n in N_TRIALS_LIST if n <= max_train]
print(f"Effective n_trials list (after trimming to data size): {N_TRIALS_LIST}")

## 3. Run Trial Count Sweep

In [ ]:
print(f"Running trial count sweep for subject '{SUBJECT_NAME}', pipeline '{PIPELINE_NAME}'")
print(f"n_trials_list = {N_TRIALS_LIST}, n_repeats = {N_REPEATS}\n")

sweep_results = run_trial_count_sweep_cv(
    epochs=epochs,
    epochs_cropped=epochs_cropped,
    cv_split=cv_split,
    params_dict=params_dict,
    n_trials_list=N_TRIALS_LIST,
    n_repeats=N_REPEATS,
    tmin=TMIN,
    tmax=TMAX,
)

print("\nSweep complete.")

## 4. Summary Table

In [ ]:
import pandas as pd

rows = []
prev_acc = None
for n in sorted(sweep_results):
    r = sweep_results[n]
    mean = r['mean_acc']
    gain = (mean - prev_acc) if prev_acc is not None and not np.isnan(mean) else float('nan')
    rows.append({
        'n_trials': n,
        'mean_acc': f"{mean:.3f}",
        'sem_acc':  f"{r['sem_acc']:.3f}",
        'std_acc':  f"{r['std_acc']:.3f}",
        'n_observations': len(r['all_accs']),
        'gain_vs_prev':   f"{gain:+.3f}" if not np.isnan(gain) else '—',
    })
    prev_acc = mean

df_summary = pd.DataFrame(rows)
print(df_summary.to_string(index=False))

## 5. Save Results

In [ ]:
save_dict = {
    'subject_name': SUBJECT_NAME,
    'pipeline_name': PIPELINE_NAME,
    'n_trials_list': N_TRIALS_LIST,
    'n_repeats': N_REPEATS,
    'tmin': TMIN,
    'tmax': TMAX,
    'sweep_results': {
        str(k): {kk: vv if not isinstance(vv, float) or not np.isnan(vv) else None
                 for kk, vv in v.items()}
        for k, v in sweep_results.items()
    }
}

classes_str = '_'.join([e[:2] for e in params_dict['desired_events']])
out_path = METRICS_DIR / f'trial_count_sweep_{SUBJECT_NAME}_{classes_str}_{PIPELINE_NAME}.json'
with open(out_path, 'w') as f:
    json.dump(save_dict, f, indent=2)

print(f"Saved to: {out_path}")

## 6. Plot Learning Curve

In [ ]:
%matplotlib inline

n_classes = len(params_dict['desired_events'])

plot_trial_count_sweep(
    sweep_results=sweep_results,
    n_classes=n_classes,
    title=f"Trial Count Sweep — {SUBJECT_NAME} / {PIPELINE_NAME}",
)

## 7. (Optional) Box-plots per Trial Count

Shows the full distribution of per-fold accuracy at each trial count,
giving a sense of variance beyond the mean ± SEM.

In [ ]:
n_trials_sorted = sorted(k for k in sweep_results if sweep_results[k]['all_accs'])
all_accs_list = [sweep_results[n]['all_accs'] for n in n_trials_sorted]

fig, ax = plt.subplots(figsize=(10, 5))
ax.boxplot(all_accs_list, labels=n_trials_sorted, patch_artist=True,
           boxprops=dict(facecolor='#008080', alpha=0.5),
           medianprops=dict(color='k', linewidth=2))

if n_classes is not None:
    ax.axhline(1.0 / n_classes, linestyle='-.', color='gray',
               label=f'Chance ({1/n_classes:.2f})')

ax.set_xlabel('Number of Training Trials', fontsize=12)
ax.set_ylabel('Classification Accuracy', fontsize=12)
ax.set_title(f'Accuracy Distribution per Trial Count — {SUBJECT_NAME}', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.4, axis='y')
plt.tight_layout()
plt.show()